# AI-Image Narration Studio - Colab

This notebook runs the **production** narration pipeline, not a demo renderer:

`script -> Piper narration -> character bible -> canonical reference images -> reference-conditioned SDXL scenes -> FFmpeg -> validation`

It has exactly two executable code cells:

1. **Setup** installs the runtime packages, locates the repository, downloads the free Piper voice, and writes the Colab `.env`.
2. **Run** executes either a real smoke test, a full video, or a resume of an interrupted job through `NarrationStudio`.

## Before you run

- Choose **Runtime > Change runtime type > T4 GPU** (16 GB). An L4 or A10G also works.
- Allow internet access. The first run downloads SDXL base (~7 GB) and IP-Adapter Plus (~0.7 GB) into the Hugging Face cache. Keep the runtime alive; later runs reuse the cache.
- The repository must be reachable at `REPO_URL`, or already uploaded to `REPO_DIR`. If you are testing uncommitted local changes, upload/clone that working copy instead of the public branch.
- Free Colab can disconnect. The pipeline resumes by job id and reuses every valid reference/scene image.

## What is real here

- **Real** character bible persistence (`characters/character_bible.json`).
- **Real** canonical reference images and IP-Adapter conditioning.
- **Real** semantic scene prompts tied to the aligned narration.
- **Real** Piper narration, FFmpeg render, stream/duration validation, and a separate 1280x720 thumbnail.
- **Not automatic** identity perfection. IP-Adapter improves continuity; diffusion models can still drift across hundreds of shots. The report includes advisory CLIP identity scores and quality flags.
- **Text LLM optional but required for topic-only full videos.** Smoke mode and `SCRIPT_PATH` full runs need no LLM key. A topic-only full run needs an OpenRouter/Gemini key so a real script can be written. With no key, the pipeline uses a clearly labelled deterministic fallback cast and fallback image prompts. Text models never generate images.

## Outputs

Each job is written under `jobs/<job_id>/`:

- `output/rendered.mp4`
- `thumbnail/thumbnail.png` (1280x720 with deterministic text)
- `manifest/manifest.json`
- `characters/character_bible.json` and `characters/<id>/reference_*.png`
- `images/scene_*.png`
- `metadata.json`

The run cell also builds a small ZIP with the video, thumbnail, manifest, bible, and metadata.


In [ ]:
\
#@title 1) Runtime setup: repository, GPU packages, Piper voice, Colab .env
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/Akashokshelke96/focusProtocol.git"  #@param {type:"string"}
REPO_REF = "main"  #@param {type:"string"}
REPO_DIR = "/content/focusProtocol"  #@param {type:"string"}
INSTALL_PACKAGES = True  #@param {type:"boolean"}
RENDER_WIDTH = 1280  #@param [1280, 1920] {type:"raw"}
RENDER_HEIGHT = 720  #@param [720, 1080] {type:"raw"}
RENDER_FPS = 15  #@param [15, 24, 30] {type:"raw"}
HF_TOKEN = ""  #@param {type:"string"}
NARRATION_SEED = 0  #@param {type:"integer"}


def run(command, *, check=True, capture=False):
    print("$", " ".join(str(part) for part in command))
    return subprocess.run(
        [str(part) for part in command],
        check=check,
        text=True,
        capture_output=capture,
    )


def write_env(path: Path, values: dict) -> None:
    lines = [
        "# Generated by deploy/narration_ai_studio_colab.ipynb",
        "# Edit and rerun the run cell for a new topic; the model cache is reused.",
    ]
    lines.extend(f"{key}={value}" for key, value in values.items())
    path.write_text("\n".join(lines) + "\n", encoding="utf-8")


try:
    import torch
except ImportError as error:
    raise RuntimeError(
        "PyTorch is missing. In Colab choose Runtime > Change runtime type > T4 GPU, "
        "then rerun this cell."
    ) from error

if not torch.cuda.is_available():
    raise RuntimeError(
        "No CUDA GPU detected. Choose Runtime > Change runtime type > T4 GPU and rerun."
    )
GPU_NAME = torch.cuda.get_device_name(0)
GPU_VRAM_GB = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
free_disk_gb = shutil.disk_usage("/content").free / (1024 ** 3)
print(f"GPU: {GPU_NAME} ({GPU_VRAM_GB:.1f} GB VRAM) | free disk: {free_disk_gb:.1f} GB")
if GPU_VRAM_GB < 14:
    raise RuntimeError("This SDXL + IP-Adapter profile expects at least 14 GB VRAM.")
if free_disk_gb < 18:
    raise RuntimeError("Free at least 18 GB in the Colab runtime, then rerun this cell.")

repo = Path(REPO_DIR).expanduser().resolve()
if not (repo / "app" / "narration" / "studio.py").is_file():
    if repo.exists() and any(repo.iterdir()):
        raise RuntimeError(
            f"{repo} exists but is not this project. Set REPO_DIR to the uploaded project "
            "folder, or delete the partial clone and rerun."
        )
    if not REPO_URL:
        raise RuntimeError(
            f"Project not found at {repo}. Upload the repository there, or set REPO_URL."
        )
    run(["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, repo])
os.chdir(repo)
# Absolute-path scripts put deploy/ on sys.path, not the repository root.
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))
print("Repository:", repo)

if INSTALL_PACKAGES:
    run(["apt-get", "-qq", "update"])
    run(["apt-get", "-qq", "install", "-y", "ffmpeg", "fonts-dejavu-core"])
    run([sys.executable, "-m", "pip", "install", "-q", "-r",
         "deploy/requirements-narration-studio.txt"])

import diffusers
import transformers
from huggingface_hub import hf_hub_download
from PIL import Image

print(f"torch {torch.__version__} | diffusers {diffusers.__version__} "
      f"| transformers {transformers.__version__}")

voice_dir = Path("/content/piper_voices")
voice_dir.mkdir(parents=True, exist_ok=True)
voice_files = {
    "model": "en/en_US/lessac/medium/en_US-lessac-medium.onnx",
    "config": "en/en_US/lessac/medium/en_US-lessac-medium.onnx.json",
}
voice_paths = {}
for key, filename in voice_files.items():
    voice_paths[key] = hf_hub_download(
        repo_id="rhasspy/piper-voices",
        filename=filename,
        local_dir=str(voice_dir),
        token=HF_TOKEN or None,
    )
model_path = Path(voice_paths["model"]).resolve()
config_path = Path(voice_paths["config"]).resolve()
if not model_path.is_file() or model_path.stat().st_size < 1_000_000:
    raise RuntimeError(f"Piper voice download failed: {model_path}")

env_path = repo / ".env"
if env_path.is_file():
    backup = repo / ".env.colab-backup"
    if not backup.exists():
        shutil.copy2(env_path, backup)
        print("Existing .env backed up to", backup)

values = {
    "VIDEO_WIDTH": RENDER_WIDTH,
    "VIDEO_HEIGHT": RENDER_HEIGHT,
    "VIDEO_FPS": RENDER_FPS,
    "FFMPEG_PRESET": "ultrafast",
    "FFMPEG_CRF": 23,
    "NARRATION_TARGET_DURATION_MINUTES": 10,
    "NARRATION_SCENE_SECONDS": 2.5,
    "NARRATION_MIN_SCENES": 1,
    "NARRATION_MAX_SCENES": 6000,
    "NARRATION_MANIFEST_BATCH_SIZE": 20,
    "NARRATION_MAX_RETRIES": 2,
    "NARRATION_IMAGE_BACKEND": "diffusers",
    "NARRATION_IMAGE_MODEL": "stabilityai/stable-diffusion-xl-base-1.0",
    "NARRATION_IMAGE_DEVICE": "cuda",
    "NARRATION_IMAGE_DTYPE": "float16",
    "NARRATION_IMAGE_CPU_OFFLOAD": "true",
    "NARRATION_IMAGE_WIDTH": 1024,
    "NARRATION_IMAGE_HEIGHT": 576,
    "NARRATION_IMAGE_STEPS": 26,
    "NARRATION_REFERENCE_WIDTH": 832,
    "NARRATION_REFERENCE_HEIGHT": 1216,
    "NARRATION_REFERENCE_STEPS": 30,
    "NARRATION_IP_ADAPTER_SCALE": 0.68,
    "NARRATION_TTS_BACKEND": "piper_local",
    "NARRATION_PIPER_BINARY": "piper",
    "NARRATION_PIPER_MODEL": str(model_path),
    "NARRATION_PIPER_CONFIG": str(config_path),
    # Piper's natural pace (~170 wpm) undershoots the requested minutes;
    # 1.17 measured ~9.3 min for a 1,444-word script (from ~8.5 min).
    "NARRATION_PIPER_LENGTH_SCALE": 1.17,
    "NARRATION_ALIGN_BACKEND": "http",
    "NARRATION_ALIGN_URL": "",
    "NARRATION_EXPORT_ENABLED": "false",
    "NARRATION_MANIFEST_REUSE": "true",
    "NARRATION_SEED": int(NARRATION_SEED),
    "NARRATION_LLM_PROVIDER": "auto",
    "CHANNEL_BRAND_NAME": "Focus Protocol",
    "HF_TOKEN": HF_TOKEN,
}
write_env(env_path, values)

from app.config.settings import Settings
from app.narration.image_backends import build_image_backend

health_settings = Settings()
backend = build_image_backend(health_settings)
try:
    health = backend.health()
finally:
    backend.close()
print(json.dumps(health, indent=2))
if not health.get("reachable", False):
    raise RuntimeError(f"Image backend is not usable: {health.get('error', health)}")

print("\nSetup complete. Continue with the run cell.")
print(f"Piper model: {model_path}")
print(f"Working directory: {Path.cwd()}")


In [ ]:
\
#@title 2) Run: real smoke test, full video, or resume
import json
import os
import shutil
import sys
from pathlib import Path
from zipfile import ZIP_DEFLATED, ZipFile

from IPython.display import FileLink, Image as DisplayImage, Video, display

RUN_MODE = "smoke"  #@param ["smoke", "full", "resume"]
TOPIC = "What did ancient humans actually do all day?"  #@param {type:"string"}
DURATION_MINUTES = 10.0  #@param {type:"number"}
SMOKE_SECONDS = 30  #@param {type:"integer"}
RESUME_JOB_ID = ""  #@param {type:"string"}
SCRIPT_PATH = ""  #@param {type:"string"}
OPENROUTER_API_KEY = ""  #@param {type:"string"}
MAKE_ZIP = True  #@param {type:"boolean"}

if OPENROUTER_API_KEY:
    os.environ["OPENROUTER_API_KEY_NARRATOR"] = OPENROUTER_API_KEY
    os.environ["NARRATION_LLM_PROVIDER"] = "openrouter"

# Keep the repository root importable even if this cell is rerun alone.
if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))

from app.narration.studio import NarrationStudio

studio = NarrationStudio()
mode = RUN_MODE.strip().lower()

if mode == "smoke":
    report = studio.smoke_test(TOPIC, SMOKE_SECONDS)
elif mode == "full":
    script_path = Path(SCRIPT_PATH.strip()).expanduser() if SCRIPT_PATH.strip() else None
    if script_path is not None:
        if not script_path.is_file():
            raise FileNotFoundError(f"SCRIPT_PATH does not exist: {script_path}")
        text = script_path.read_text(encoding="utf-8").strip()
        if len(text.split()) < 250:
            raise ValueError(
                "The supplied script is too short for a full video. A 10-minute video "
                "needs roughly 1,300-1,650 spoken words.")
        from app.narration.models import ScriptPlan
        plan = ScriptPlan(
            topic=TOPIC,
            title=TOPIC[:90],
            script=text,
            word_count=len(text.split()),
            estimated_duration_seconds=round(len(text.split()) / 150.0 * 60, 2),
            approved=True,
        )
        ok = studio.run_script(plan, DURATION_MINUTES)
    else:
        from app.config.settings import Settings
        from app.narration.llm import build_narration_llm
        if not build_narration_llm(Settings()).available:
            raise RuntimeError(
                "Full mode needs a text LLM key to write the script, or set SCRIPT_PATH "
                "to an approved narration .txt file. Smoke mode does not need an LLM key.")
        ok = studio.run_topic(TOPIC, DURATION_MINUTES, auto_approve=True)
    report = studio.build_report(studio.last_job_id)
    report["ok"] = bool(ok and report.get("checks", {}).get("all_passed"))
elif mode == "resume":
    if not RESUME_JOB_ID.strip():
        raise ValueError("RESUME_JOB_ID is required when RUN_MODE is 'resume'")
    ok = studio.resume(RESUME_JOB_ID.strip())
    report = studio.build_report(RESUME_JOB_ID.strip())
    report["ok"] = bool(ok and report.get("checks", {}).get("all_passed"))
else:
    raise ValueError(f"Unknown RUN_MODE: {RUN_MODE}")

print(json.dumps(report, indent=2, default=str))
job_id = report.get("job_id") or studio.last_job_id or ""
job_dir = Path("jobs") / job_id
video_path = Path(report.get("video") or job_dir / "output" / "rendered.mp4")
thumbnail_path = Path(report.get("thumbnail") or job_dir / "thumbnail" / "thumbnail.png")

if MAKE_ZIP and job_id and video_path.is_file():
    package = Path("/content") / f"{job_id}_package.zip"
    wanted = [
        video_path,
        thumbnail_path,
        job_dir / "metadata.json",
        job_dir / "manifest" / "manifest.json",
        job_dir / "characters" / "character_bible.json",
    ]
    with ZipFile(package, "w", ZIP_DEFLATED) as archive:
        for path in wanted:
            if path.is_file():
                archive.write(path, arcname=str(path.relative_to(job_dir)))
    print("Download package:", package)
    display(FileLink(str(package)))

if video_path.is_file():
    display(Video(str(video_path), embed=True, width=960))
else:
    print("No rendered MP4 was produced. Read the report's validation/errors above.")
if thumbnail_path.is_file():
    display(DisplayImage(filename=str(thumbnail_path)))

if report.get("status") == "completed_with_fallback":
    print(
        "Job completed with a fallback. This is expected without a text LLM key "
        "(deterministic cast/prompts) or when WhisperX alignment is unavailable "
        "(estimated alignment). The reference images and scene images are still real."
    )
